In [ ]:
# ==================== Fragment Merging 评估流程 ====================
# 评估流程：
# 1. 有效性筛选：剔除无法转化为RDKit Mol对象、电荷异常或结构碎片化的无效样本
# 2. xTB几何优化：确保构象稳定
# 3. 两个维度评估：
#    a. ESP相似度和药效团相似度（与条件数据最优对齐）
#    b. SA分数筛选（SA≤4.0且中性电荷）

import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "0"

import json
import pickle
import sys
from datetime import datetime
from typing import Optional, Tuple, List, Dict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem, QED, Crippen, Lipinski, Descriptors
from tqdm import tqdm

from importlib.metadata import distributions
if any(d.metadata["Name"] == 'rdkit' for d in distributions()):
    from rdkit.Contrib.SA_Score import sascorer
else:
    sys.path.append(os.path.join(os.environ['CONDA_PREFIX'],'share','RDKit','Contrib'))
    from SA_Score import sascorer

from lightning_fabric.utilities.seed import seed_everything
seed_everything(42)

# Shepherd相关模块
from shepherd.extract import create_rdkit_molecule_from_mol
from shepherd.shepherd_score_utils.conformer_generation import update_mol_coordinates
from shepherd.shepherd_score_utils.generate_point_cloud import (
    get_atomic_vdw_radii, 
    get_molecular_surface,
    get_electrostatics_given_point_charges,
)
from shepherd.shepherd_score_utils.pharm_utils.pharmacophore import get_pharmacophores

# Shepherd Score模块
from shepherd_score.container import Molecule, MoleculePair
from shepherd_score.conformer_generation import optimize_conformer_with_xtb_from_xyz_block, single_point_xtb_from_xyz
from shepherd_score.evaluations.utils.convert_data import extract_mol_from_xyz_block, get_mol_from_atom_pos
from shepherd_score.score.constants import ALPHA, LAM_SCALING
from shepherd_score.score.gaussian_overlap_np import get_overlap_np
from shepherd_score.score.electrostatic_scoring_np import get_overlap_esp_np
from shepherd_score.score.pharmacophore_scoring_np import get_overlap_pharm_np

print("✅ 所有模块导入完成")

Seed set to 0


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


# ==================== 配置参数 ====================

# 数据路径
CONDITION_FILE = '/home1/zhh/workspace/SPD/data/conformers/fragment_merging/fragment_merge_condition.pickle'
SAMPLES_DIR = '/home1/zhh/workspace/SPD/evaluation/core_data/data/3'

# 评估参数
NUM_SURF_POINTS = 75  # 与条件数据一致
PROBE_RADIUS = 1.2     # 探针半径
PHARM_MULTI_VECTOR = True  # 药效团多向量模式
SOLVENT = 'water'      # xTB溶剂模型

# SA分数筛选阈值
SA_THRESHOLD = 4.0

# 临时文件目录
TMPDIR = Path('/tmp/xtb_eval')
TMPDIR.mkdir(parents=True, exist_ok=True)

print(f"📁 条件数据: {CONDITION_FILE}")
print(f"📁 样本目录: {SAMPLES_DIR}")
print(f"📊 表面点数: {NUM_SURF_POINTS}")
print(f"📊 SA阈值: {SA_THRESHOLD}")

In [ ]:
# ==================== 加载条件数据 ====================

with open(CONDITION_FILE, 'rb') as f:
    condition_data = pickle.load(f)

# 提取条件信息
cond_surface_pos = condition_data['x3']['positions']  # ESP表面位置
cond_surface_esp = condition_data['x3']['charges']    # ESP电荷值
cond_pharm_types = condition_data['x4']['types']      # 药效团类型
cond_pharm_pos = condition_data['x4']['positions']    # 药效团位置
cond_pharm_dir = condition_data['x4']['directions']   # 药效团方向

print(f"{'='*60}")
print("? 条件数据结构:")
print(f"  - ESP表面点数: {len(cond_surface_pos)}")
print(f"  - ESP表面位置: {cond_surface_pos.shape}")
print(f"  - ESP电荷值: {cond_surface_esp.shape}")
print(f"  - 药效团数量: {len(cond_pharm_types)}")
print(f"  - 药效团类型分布: {np.bincount(cond_pharm_types.astype(int))}")
print(f"{'='*60}")

✅ DPO加载 generated_mols_20260117_151541.json: 12 个样本
✅ DPO加载 generated_mols_20260117_160607.json: 12 个样本
✅ DPO加载 generated_mols_20260117_154138.json: 12 个样本
✅ DPO加载 generated_mols_20260117_171005.json: 12 个样本
✅ DPO加载 generated_mols_20260117_164500.json: 12 个样本
📊 DPO模型: 总共 60 个样本
   - 参考分子0: 20 个样本
   - 参考分子1: 20 个样本
   - 参考分子2: 20 个样本
📊 原始Shepherd模型: 总共 60 个样本
   - 参考分子0: 20 个样本
   - 参考分子1: 20 个样本
   - 参考分子2: 20 个样本
📊 自训练SPD模型: 总共 60 个样本
   - 参考分子0: 20 个样本
   - 参考分子1: 20 个样本
   - 参考分子2: 20 个样本

📋 三种模型采样数据加载并分组完成:
  - DPO: 60 个样本
      参考分子0: 20 个
      参考分子1: 20 个
      参考分子2: 20 个
  - Origin_Shepherd: 60 个样本
      参考分子0: 20 个
      参考分子1: 20 个
      参考分子2: 20 个
  - SPD: 60 个样本
      参考分子0: 20 个
      参考分子1: 20 个
      参考分子2: 20 个


In [ ]:
# ==================== 加载采样数据 ====================

def convert_sample_format(sample):
    """将JSON数据转换为numpy数组格式"""
    modal_keys = ['x1', 'x2', 'x3', 'x4']
    for modal_key in modal_keys:
        if modal_key in sample and isinstance(sample[modal_key], dict):
            for data_key in sample[modal_key]:
                if isinstance(sample[modal_key][data_key], list):
                    sample[modal_key][data_key] = np.array(sample[modal_key][data_key])
    return sample

def load_fragment_merge_samples(json_path):
    """加载fragment_merge采样数据"""
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    all_samples = []
    results = data.get('results', {})
    
    for n_atoms_key in sorted(results.keys(), key=lambda x: int(x.split('_')[-1])):
        n_atoms_data = results[n_atoms_key]
        samples = n_atoms_data.get('samples', [])
        for sample in samples:
            sample = convert_sample_format(sample)
            all_samples.append(sample)
    
    return all_samples, data

# 加载所有模型的采样数据
all_model_samples = {}

# 检查各模型目录
model_dirs = {
    'Shepherd_80_50': os.path.join(SAMPLES_DIR, 'shepherd80_50', 'fragment_merge_samples.json'),
}

# 检查DPO和SPD目录
dpo_dir = os.path.join(SAMPLES_DIR, 'DPO')
spd_dir = os.path.join(SAMPLES_DIR, 'SPD')

if os.path.exists(dpo_dir) and os.listdir(dpo_dir):
    dpo_files = [f for f in os.listdir(dpo_dir) if f.endswith('.json')]
    if dpo_files:
        model_dirs['DPO'] = os.path.join(dpo_dir, dpo_files[0])

if os.path.exists(spd_dir) and os.listdir(spd_dir):
    spd_files = [f for f in os.listdir(spd_dir) if f.endswith('.json')]
    if spd_files:
        model_dirs['SPD'] = os.path.join(spd_dir, spd_files[0])

print(f"{'='*60}")
print("? 加载采样数据:")

for model_name, json_path in model_dirs.items():
    if os.path.exists(json_path):
        samples, metadata = load_fragment_merge_samples(json_path)
        all_model_samples[model_name] = samples
        print(f"  ✅ {model_name}: {len(samples)} 个样本")
        print(f"     文件: {json_path}")
    else:
        print(f"  ⚠️ {model_name}: 文件不存在 - {json_path}")

print(f"{'='*60}")
print(f"📊 总计加载 {len(all_model_samples)} 个模型的数据")


🔬 开始评估 DPO 模型的 60 个样本
正在评估第 1/60 个生成结构...
  ✓ 第 1 个结构评估完成: QED=0.424, SA_score=4.329
正在评估第 2/60 个生成结构...
  ✓ 第 2 个结构评估完成: QED=0.545, SA_score=4.821
正在评估第 3/60 个生成结构...
  ✓ 第 3 个结构评估完成: QED=N/A, SA_score=N/A
正在评估第 4/60 个生成结构...
  ✓ 第 4 个结构评估完成: QED=0.285, SA_score=5.239
正在评估第 5/60 个生成结构...
  ✓ 第 5 个结构评估完成: QED=0.716, SA_score=4.958
正在评估第 6/60 个生成结构...
  ✓ 第 6 个结构评估完成: QED=0.654, SA_score=4.834
正在评估第 7/60 个生成结构...
  ✓ 第 7 个结构评估完成: QED=0.648, SA_score=4.803
正在评估第 8/60 个生成结构...
  ✓ 第 8 个结构评估完成: QED=0.562, SA_score=4.079
正在评估第 9/60 个生成结构...


[10:10:18] non-ring atom 11 marked aromatic
[10:10:18] non-ring atom 11 marked aromatic
[10:10:18] Can't kekulize mol.  Unkekulized atoms: 19


  ✗ 第 9 个结构评估失败: non-ring atom 11 marked aromatic
正在评估第 10/60 个生成结构...


[10:10:19] non-ring atom 18 marked aromatic


  ✗ 第 10 个结构评估失败: non-ring atom 18 marked aromatic
正在评估第 11/60 个生成结构...
  ✓ 第 11 个结构评估完成: QED=0.102, SA_score=4.749
正在评估第 12/60 个生成结构...


...

  ✓ 第 52 个结构评估完成: QED=N/A, SA_score=N/A
正在评估第 53/60 个生成结构...
  ✓ 第 53 个结构评估完成: QED=0.505, SA_score=4.158
正在评估第 54/60 个生成结构...


[10:39:35] Explicit valence for atom # 23 C, 5, is greater than permitted
[10:39:35] Explicit valence for atom # 23 C, 5, is greater than permitted
[10:39:35] Explicit valence for atom # 23 C, 5, is greater than permitted
[10:39:35] Explicit valence for atom # 23 C, 5, is greater than permitted
[10:39:35] Explicit valence for atom # 23 C, 5, is greater than permitted
[10:39:35] Explicit valence for atom # 23 C, 5, is greater than permitted
[10:39:35] Explicit valence for atom # 23 C, 5, is greater than permitted
[10:39:35] Explicit valence for atom # 23 C, 5, is greater than permitted
[10:39:35] Explicit valence for atom # 23 C, 5, is greater than permitted
[10:39:35] Explicit valence for atom # 23 C, 5, is greater than permitted
[10:39:53] Explicit valence for atom # 23 C, 5, is greater than permitted
[10:39:53] Explicit valence for atom # 23 C, 5, is greater than permitted


  ✓ 第 54 个结构评估完成: QED=N/A, SA_score=N/A
正在评估第 55/60 个生成结构...
  ✓ 第 55 个结构评估完成: QED=0.282, SA_score=4.234
正在评估第 56/60 个生成结构...
  ✓ 第 56 个结构评估完成: QED=0.193, SA_score=4.950
正在评估第 57/60 个生成结构...
  ✓ 第 57 个结构评估完成: QED=0.251, SA_score=4.592
正在评估第 58/60 个生成结构...


[10:40:44] Explicit valence for atom # 9 C, 5, is greater than permitted
[10:40:44] Explicit valence for atom # 9 C, 5, is greater than permitted
[10:40:44] Explicit valence for atom # 9 C, 5, is greater than permitted
[10:40:44] Explicit valence for atom # 9 C, 5, is greater than permitted
[10:40:44] Explicit valence for atom # 9 C, 5, is greater than permitted
[10:40:44] Explicit valence for atom # 9 C, 5, is greater than permitted
[10:40:44] Explicit valence for atom # 9 C, 5, is greater than permitted
[10:40:44] Explicit valence for atom # 9 C, 5, is greater than permitted
[10:40:44] Explicit valence for atom # 9 C, 5, is greater than permitted
[10:40:44] Explicit valence for atom # 9 C, 5, is greater than permitted
[10:40:56] Explicit valence for atom # 9 C, 5, is greater than permitted
[10:40:56] Explicit valence for atom # 9 C, 5, is greater than permitted


  ✓ 第 58 个结构评估完成: QED=N/A, SA_score=N/A
正在评估第 59/60 个生成结构...
  ✓ 第 59 个结构评估完成: QED=N/A, SA_score=N/A
正在评估第 60/60 个生成结构...
  ✓ 第 60 个结构评估完成: QED=0.432, SA_score=4.691

✅ SPD 模型: 成功评估 57/60 个结构

📊 所有模型ConfEval评估完成!

💾 评估结果已保存到: conf_eval_results_20260123_104123.json
   - 文件包含 3 个模型的评估数据
   - DPO: 60 个样本
   - Origin_Shepherd: 60 个样本
   - SPD: 60 个样本


In [ ]:
# ==================== 有效性筛选函数 ====================

def check_validity(atoms, positions, bonds=None):
    """
    检查分子有效性：
    1. 能否转化为RDKit Mol对象
    2. 电荷是否异常
    3. 是否存在结构碎片化
    
    Returns:
        tuple: (is_valid, mol, charge, xyz_block, error_msg)
    """
    try:
        # 转换为RDKit Mol对象
        mol, charge, xyz_block = get_mol_from_atom_pos(
            atoms=atoms, 
            positions=positions, 
            bonds=bonds
        )
        
        if mol is None:
            return False, None, None, None, "无法转化为RDKit Mol对象"
        
        # 检查电荷是否异常（中性或合理范围内）
        formal_charge = Chem.GetFormalCharge(mol)
        if abs(formal_charge) > 2:
            return False, mol, charge, xyz_block, f"电荷异常: {formal_charge}"
        
        # 检查结构碎片化（是否只有一个连通分量）
        mol_no_h = Chem.RemoveHs(mol)
        frags = Chem.GetMolFrags(mol_no_h)
        if len(frags) > 1:
            return False, mol, charge, xyz_block, f"结构碎片化: {len(frags)} 个片段"
        
        # 检查是否有原子
        if mol.GetNumAtoms() == 0:
            return False, None, None, None, "分子无原子"
        
        return True, mol, charge, xyz_block, None
        
    except Exception as e:
        return False, None, None, None, f"验证异常: {str(e)}"


def filter_valid_samples(samples, model_name):
    """
    对采样分子进行有效性筛选
    
    Returns:
        tuple: (valid_samples, invalid_samples, stats)
    """
    valid_samples = []
    invalid_samples = []
    
    error_counts = {
        '无法转化为RDKit Mol对象': 0,
        '电荷异常': 0,
        '结构碎片化': 0,
        '验证异常': 0,
        '分子无原子': 0,
    }
    
    print(f"\n? 筛选 {model_name} 模型的 {len(samples)} 个样本...")
    
    for i, sample in enumerate(tqdm(samples, desc=f"有效性筛选")):
        atoms = sample['x1']['atoms']
        positions = sample['x1']['positions']
        bonds = sample['x1'].get('bonds', None)
        
        # 处理原子数组格式
        if isinstance(atoms, np.ndarray):
            atoms = atoms.flatten()
        
        is_valid, mol, charge, xyz_block, error_msg = check_validity(atoms, positions, bonds)
        
        if is_valid:
            sample['_mol'] = mol
            sample['_charge'] = charge
            sample['_xyz_block'] = xyz_block
            sample['_sample_idx'] = i
            valid_samples.append(sample)
        else:
            # 统计错误类型
            for error_type in error_counts.keys():
                if error_msg and error_type in error_msg:
                    error_counts[error_type] += 1
                    break
            invalid_samples.append({
                'sample_idx': i,
                'error': error_msg,
                'n_atoms': sample.get('n_atoms', len(atoms))
            })
    
    stats = {
        'total': len(samples),
        'valid': len(valid_samples),
        'invalid': len(invalid_samples),
        'validity_rate': len(valid_samples) / len(samples) * 100 if samples else 0,
        'error_breakdown': error_counts
    }
    
    return valid_samples, invalid_samples, stats

print("✅ 有效性筛选函数定义完成")

🔍 找到评估结果文件: conf_eval_results_20260123_104123.json

📋 文件包含的模型: ['DPO', 'Origin_Shepherd', 'SPD']

🔬 DPO 模型的评估指标:

样本ID: 0 - 包含 29 个评估指标:
------------------------------------------------------------

📊 数值型指标:
  - QED: 0.424322
  - QED_post_opt: 0.424322
  - SA_score: 4.328859
  - SA_score_post_opt: 4.328859
  - charge: 0
  - energy: -100.277398
  - energy_post_opt: -100.323233
  - fsp3: 0.818182
  - fsp3_post_opt: 0.818182
  - is_graph_consistent: True
  - is_valid: True
  - is_valid_post_opt: True
  - logP: -0.003100
  - logP_post_opt: -0.003100
  - rmsd: 1.150354
  - strain_energy: 0.045835

📝 字符串指标:
  - mol: <rdkit.Chem.rdchem.Mol object at 0x7f0107873ac0>
  - mol_post_opt: <rdkit.Chem.rdchem.Mol object at 0x7f0107873e40>
  - solvent: water

🔧 其他类型指标:
  - molblock: [str]
  - molblock_post_opt: [str]
  - morgan_fp: [str]
  - morgan_fp_post_opt: [str]
  - partial_charges: [list]
  - partial_charges_post_opt: [list]
  - smiles: [str]
  - smiles_post_opt: [str]
  - xyz_block: [str]
  - x

In [ ]:
# ==================== xTB几何优化函数 ====================

def optimize_with_xtb(valid_samples, model_name):
    """
    对有效分子进行xTB几何优化
    
    Returns:
        tuple: (optimized_samples, failed_samples, stats)
    """
    optimized_samples = []
    failed_samples = []
    
    print(f"\n⚙️ xTB优化 {model_name} 模型的 {len(valid_samples)} 个有效样本...")
    
    for sample in tqdm(valid_samples, desc="xTB几何优化"):
        xyz_block = sample['_xyz_block']
        charge = sample['_charge']
        sample_idx = sample['_sample_idx']
        
        try:
            # xTB几何优化
            xtb_result = optimize_conformer_with_xtb_from_xyz_block(
                xyz_block,
                solvent=SOLVENT,
                num_cores=1,
                charge=charge,
                temp_dir=TMPDIR
            )
            
            if xtb_result is None:
                failed_samples.append({
                    'sample_idx': sample_idx,
                    'error': 'xTB优化返回None'
                })
                continue
            
            xyz_block_opt, energy_opt, partial_charges_opt = xtb_result
            
            if xyz_block_opt is None or partial_charges_opt is None:
                failed_samples.append({
                    'sample_idx': sample_idx,
                    'error': 'xTB优化结果不完整'
                })
                continue
            
            # 解析优化后的坐标
            positions_opt = None
            lines = xyz_block_opt.strip().split('\n')
            if len(lines) > 2:
                coords = []
                for line in lines[2:]:
                    parts = line.split()
                    if len(parts) >= 4:
                        coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
                positions_opt = np.array(coords)
            
            # 获取原子信息
            atoms = sample['x1']['atoms']
            if isinstance(atoms, np.ndarray):
                atoms = atoms.flatten()
            if len(atoms.shape) == 2:
                atomic_nums = np.argmin(np.abs(atoms - 1.0), axis=-1)
            else:
                atomic_nums = atoms
            
            bonds = sample['x1'].get('bonds', None)
            
            # 从优化后的xyz_block提取mol
            mol_opt = extract_mol_from_xyz_block(
                xyz_block=xyz_block_opt,
                charge=charge,
                atoms=atomic_nums,
                positions=positions_opt,
                bonds=bonds
            )
            
            if mol_opt is None:
                failed_samples.append({
                    'sample_idx': sample_idx,
                    'error': '无法从优化后xyz提取Mol对象'
                })
                continue
            
            # 过滤partial_charges以匹配非氢原子
            partial_charges_filtered = np.array(partial_charges_opt)
            non_h_mask = (atomic_nums > 1) & (atomic_nums <= 118)
            if len(partial_charges_filtered) == len(atomic_nums):
                partial_charges_filtered = partial_charges_filtered[non_h_mask]
            
            # 保存优化结果
            sample['_mol_opt'] = mol_opt
            sample['_xyz_block_opt'] = xyz_block_opt
            sample['_energy_opt'] = energy_opt
            sample['_partial_charges_opt'] = partial_charges_filtered
            sample['_positions_opt'] = positions_opt
            
            optimized_samples.append(sample)
            
        except Exception as e:
            failed_samples.append({
                'sample_idx': sample_idx,
                'error': f'xTB优化异常: {str(e)}'
            })
    
    stats = {
        'input': len(valid_samples),
        'optimized': len(optimized_samples),
        'failed': len(failed_samples),
        'success_rate': len(optimized_samples) / len(valid_samples) * 100 if valid_samples else 0
    }
    
    return optimized_samples, failed_samples, stats

print("✅ xTB几何优化函数定义完成")

📊 加载评估数据: /home1/zhh/workspace/SPD/evaluation/experiment/conf_eval_results_20260123_104123.json
✅ 成功加载 3 个模型的评估数据
   模型列表: ['DPO', 'Origin_Shepherd', 'SPD']

📈 1. 基础统计信息

🔹 DPO 模型:
   总样本数: 60
   初始有效: 23 (38.3%)
   优化后有效: 23 (38.3%)
   图一致性: 23 (38.3%)

🔹 Origin_Shepherd 模型:
   总样本数: 60
   初始有效: 33 (55.0%)
   优化后有效: 33 (55.0%)
   图一致性: 33 (55.0%)

🔹 SPD 模型:
   总样本数: 60
   初始有效: 33 (55.0%)
   优化后有效: 33 (55.0%)
   图一致性: 33 (55.0%)

📊 2. 关键化学性质指标统计

📌 DPO 模型指标统计:

   QED:
      样本数: 23
      平均值±标准差: 0.4608 ± 0.1581
      范围: [0.1023, 0.7162]
      四分位数: Q1=0.3774, Q2=0.4840, Q3=0.5687

   SA_score:
      样本数: 23
      平均值±标准差: 4.9490 ± 0.6339
      范围: [3.8874, 6.5690]
      四分位数: Q1=4.5404, Q2=4.9564, Q3=5.3058

   logP:
      样本数: 23
      平均值±标准差: 1.2480 ± 1.7731
      范围: [-1.2282, 5.7522]
      四分位数: Q1=-0.1871, Q2=0.9893, Q3=2.1794

   strain_energy:
      样本数: 49
      平均值±标准差: 0.1795 ± 0.1039
      范围: [0.0357, 0.5099]
      四分位数: Q1=0.1093, Q2=0.1531, Q3=0.2390

   fsp3:
      

In [ ]:
# ==================== ESP和药效团相似度评估函数 ====================

def compute_similarity_scores(optimized_samples, model_name):
    """
    计算优化后分子与条件数据的ESP相似度和药效团相似度
    使用最优对齐计算相似度
    
    Returns:
        tuple: (scored_samples, failed_samples, stats)
    """
    scored_samples = []
    failed_samples = []
    
    # 计算alpha参数
    alpha = ALPHA(NUM_SURF_POINTS)
    lam_scaled = 0.3 * LAM_SCALING
    
    print(f"\n📐 计算 {model_name} 模型的 {len(optimized_samples)} 个样本的相似度...")
    
    for sample in tqdm(optimized_samples, desc="相似度计算"):
        sample_idx = sample['_sample_idx']
        mol_opt = sample['_mol_opt']
        partial_charges = sample['_partial_charges_opt']
        
        try:
            # 创建生成分子的Molecule对象（用于计算表面和药效团）
            gen_molec = Molecule(
                mol_opt,
                num_surf_points=NUM_SURF_POINTS,
                probe_radius=PROBE_RADIUS,
                partial_charges=partial_charges,
                pharm_multi_vector=PHARM_MULTI_VECTOR
            )
            
            if gen_molec.surf_pos is None:
                failed_samples.append({
                    'sample_idx': sample_idx,
                    'error': '无法生成分子表面'
                })
                continue
            
            # ========== 计算ESP相似度 ==========
            # 直接与条件ESP表面计算相似度（无需对齐，因为生成时已经以条件为中心）
            esp_sim = get_overlap_esp_np(
                gen_molec.surf_pos,      # 生成分子的表面位置
                cond_surface_pos,         # 条件ESP表面位置
                gen_molec.surf_esp,       # 生成分子的ESP
                cond_surface_esp,         # 条件ESP值
                alpha=alpha,
                lam=lam_scaled
            )
            
            # ========== 计算药效团相似度 ==========
            pharm_sim = None
            if gen_molec.pharm_ancs is not None and len(gen_molec.pharm_ancs) > 0:
                pharm_sim = get_overlap_pharm_np(
                    gen_molec.pharm_types,    # 生成分子的药效团类型
                    cond_pharm_types,          # 条件药效团类型
                    gen_molec.pharm_ancs,     # 生成分子的药效团位置
                    cond_pharm_pos,            # 条件药效团位置
                    gen_molec.pharm_vecs,     # 生成分子的药效团方向
                    cond_pharm_dir,            # 条件药效团方向
                    similarity='tanimoto',
                    extended_points=False,
                    only_extended=False
                )
            
            # 保存相似度分数
            sample['_esp_similarity'] = float(esp_sim) if esp_sim is not None else None
            sample['_pharm_similarity'] = float(pharm_sim) if pharm_sim is not None else None
            sample['_gen_molec'] = gen_molec
            
            scored_samples.append(sample)
            
        except Exception as e:
            failed_samples.append({
                'sample_idx': sample_idx,
                'error': f'相似度计算异常: {str(e)}'
            })
    
    # 统计相似度分布
    esp_sims = [s['_esp_similarity'] for s in scored_samples if s['_esp_similarity'] is not None]
    pharm_sims = [s['_pharm_similarity'] for s in scored_samples if s['_pharm_similarity'] is not None]
    
    stats = {
        'input': len(optimized_samples),
        'scored': len(scored_samples),
        'failed': len(failed_samples),
        'esp_mean': np.mean(esp_sims) if esp_sims else None,
        'esp_std': np.std(esp_sims) if esp_sims else None,
        'esp_max': np.max(esp_sims) if esp_sims else None,
        'pharm_mean': np.mean(pharm_sims) if pharm_sims else None,
        'pharm_std': np.std(pharm_sims) if pharm_sims else None,
        'pharm_max': np.max(pharm_sims) if pharm_sims else None,
    }
    
    return scored_samples, failed_samples, stats

print("✅ 相似度评估函数定义完成")

In [ ]:
# ==================== SA分数筛选和2D属性计算 ====================

def compute_2d_properties_and_filter(scored_samples, model_name):
    """
    计算2D分子属性并筛选SA≤4.0且中性电荷的分子
    
    Returns:
        tuple: (filtered_samples, all_properties, stats)
    """
    all_properties = []
    filtered_samples = []
    
    print(f"\n🧪 计算 {model_name} 模型的 {len(scored_samples)} 个样本的2D属性...")
    
    for sample in tqdm(scored_samples, desc="2D属性计算"):
        sample_idx = sample['_sample_idx']
        mol_opt = sample['_mol_opt']
        
        try:
            mol_no_h = Chem.RemoveHs(mol_opt)
            
            # 计算2D属性
            sa_score = sascorer.calculateScore(mol_no_h)
            qed_score = QED.qed(mol_opt)
            logp = Crippen.MolLogP(mol_opt)
            mw = Descriptors.MolWt(mol_opt)
            hbd = Lipinski.NumHDonors(mol_opt)
            hba = Lipinski.NumHAcceptors(mol_opt)
            rotatable_bonds = Lipinski.NumRotatableBonds(mol_opt)
            tpsa = Descriptors.TPSA(mol_opt)
            fsp3 = Lipinski.FractionCSP3(mol_opt)
            
            # 获取形式电荷
            formal_charge = Chem.GetFormalCharge(mol_opt)
            
            # 获取SMILES
            smiles = Chem.MolToSmiles(mol_no_h)
            
            # 属性字典
            props = {
                'sample_idx': sample_idx,
                'n_atoms': sample.get('n_atoms', mol_opt.GetNumAtoms()),
                'smiles': smiles,
                'SA_score': sa_score,
                'QED': qed_score,
                'logP': logp,
                'MW': mw,
                'HBD': hbd,
                'HBA': hba,
                'rotatable_bonds': rotatable_bonds,
                'TPSA': tpsa,
                'fsp3': fsp3,
                'formal_charge': formal_charge,
                'esp_similarity': sample['_esp_similarity'],
                'pharm_similarity': sample['_pharm_similarity'],
                'energy_opt': sample.get('_energy_opt', None),
            }
            
            sample['_properties'] = props
            all_properties.append(props)
            
            # 筛选条件：SA≤4.0 且 中性电荷
            if sa_score <= SA_THRESHOLD and formal_charge == 0:
                sample['_passed_filter'] = True
                filtered_samples.append(sample)
            else:
                sample['_passed_filter'] = False
            
        except Exception as e:
            print(f"  ⚠️ 样本 {sample_idx} 属性计算失败: {str(e)}")
            continue
    
    # 统计
    sa_scores = [p['SA_score'] for p in all_properties]
    qed_scores = [p['QED'] for p in all_properties]
    charges = [p['formal_charge'] for p in all_properties]
    
    neutral_count = sum(1 for c in charges if c == 0)
    sa_pass_count = sum(1 for s in sa_scores if s <= SA_THRESHOLD)
    
    stats = {
        'input': len(scored_samples),
        'computed': len(all_properties),
        'filtered': len(filtered_samples),
        'filter_rate': len(filtered_samples) / len(all_properties) * 100 if all_properties else 0,
        'sa_mean': np.mean(sa_scores) if sa_scores else None,
        'sa_std': np.std(sa_scores) if sa_scores else None,
        'sa_pass_count': sa_pass_count,
        'sa_pass_rate': sa_pass_count / len(sa_scores) * 100 if sa_scores else 0,
        'qed_mean': np.mean(qed_scores) if qed_scores else None,
        'qed_std': np.std(qed_scores) if qed_scores else None,
        'neutral_count': neutral_count,
        'neutral_rate': neutral_count / len(charges) * 100 if charges else 0,
    }
    
    return filtered_samples, all_properties, stats

print("✅ SA分数筛选和2D属性计算函数定义完成")

In [ ]:
# ==================== 完整评估流程 ====================

def run_full_evaluation(samples, model_name):
    """
    运行完整的评估流程：
    1. 有效性筛选
    2. xTB几何优化
    3. ESP和药效团相似度计算
    4. SA分数筛选
    
    Returns:
        dict: 包含所有评估结果和统计信息
    """
    print(f"\n{'='*70}")
    print(f"🚀 开始评估 {model_name} 模型")
    print(f"{'='*70}")
    
    results = {
        'model_name': model_name,
        'total_samples': len(samples),
    }
    
    # Step 1: 有效性筛选
    print(f"\n? Step 1/4: 有效性筛选")
    valid_samples, invalid_samples, validity_stats = filter_valid_samples(samples, model_name)
    results['validity_stats'] = validity_stats
    print(f"   有效率: {validity_stats['validity_rate']:.1f}% ({validity_stats['valid']}/{validity_stats['total']})")
    
    if not valid_samples:
        print(f"   ⚠️ 无有效样本，跳过后续步骤")
        return results
    
    # Step 2: xTB几何优化
    print(f"\n📍 Step 2/4: xTB几何优化")
    optimized_samples, opt_failed, opt_stats = optimize_with_xtb(valid_samples, model_name)
    results['optimization_stats'] = opt_stats
    print(f"   优化成功率: {opt_stats['success_rate']:.1f}% ({opt_stats['optimized']}/{opt_stats['input']})")
    
    if not optimized_samples:
        print(f"   ⚠️ 无优化成功样本，跳过后续步骤")
        return results
    
    # Step 3: ESP和药效团相似度计算
    print(f"\n📍 Step 3/4: 相似度计算")
    scored_samples, score_failed, score_stats = compute_similarity_scores(optimized_samples, model_name)
    results['similarity_stats'] = score_stats
    if score_stats['esp_mean'] is not None:
        print(f"   ESP相似度: {score_stats['esp_mean']:.4f} ± {score_stats['esp_std']:.4f} (max: {score_stats['esp_max']:.4f})")
    if score_stats['pharm_mean'] is not None:
        print(f"   药效团相似度: {score_stats['pharm_mean']:.4f} ± {score_stats['pharm_std']:.4f} (max: {score_stats['pharm_max']:.4f})")
    
    if not scored_samples:
        print(f"   ⚠️ 无相似度计算成功样本，跳过后续步骤")
        return results
    
    # Step 4: SA分数筛选和2D属性计算
    print(f"\n📍 Step 4/4: 2D属性计算和SA筛选")
    filtered_samples, all_properties, filter_stats = compute_2d_properties_and_filter(scored_samples, model_name)
    results['filter_stats'] = filter_stats
    results['all_properties'] = all_properties
    results['filtered_samples'] = filtered_samples
    
    print(f"   SA平均值: {filter_stats['sa_mean']:.2f} ± {filter_stats['sa_std']:.2f}")
    print(f"   SA≤{SA_THRESHOLD}通过率: {filter_stats['sa_pass_rate']:.1f}%")
    print(f"   中性电荷比例: {filter_stats['neutral_rate']:.1f}%")
    print(f"   最终筛选通过率: {filter_stats['filter_rate']:.1f}% ({filter_stats['filtered']}/{filter_stats['computed']})")
    
    # 汇总统计
    print(f"\n{'='*70}")
    print(f"📊 {model_name} 评估完成汇总:")
    print(f"   总样本数: {len(samples)}")
    print(f"   有效样本: {validity_stats['valid']} ({validity_stats['validity_rate']:.1f}%)")
    print(f"   优化成功: {opt_stats['optimized']} ({opt_stats['success_rate']:.1f}%)")
    print(f"   相似度计算: {score_stats['scored']}")
    print(f"   SA≤{SA_THRESHOLD}且中性: {filter_stats['filtered']} ({filter_stats['filter_rate']:.1f}%)")
    print(f"{'='*70}")
    
    return results

print("✅ 完整评估流程函数定义完成")

In [ ]:
# ==================== 执行评估 ====================

all_results = {}

for model_name, samples in all_model_samples.items():
    results = run_full_evaluation(samples, model_name)
    all_results[model_name] = results

print(f"\n{'='*70}")
print("🎉 所有模型评估完成!")
print(f"{'='*70}")

In [ ]:
# ==================== 结果汇总和对比 ====================

def create_summary_table(all_results):
    """创建各模型评估结果汇总表"""
    summary_data = []
    
    for model_name, results in all_results.items():
        row = {
            '模型': model_name,
            '总样本': results.get('total_samples', 0),
        }
        
        # 有效性统计
        if 'validity_stats' in results:
            v = results['validity_stats']
            row['有效样本'] = v.get('valid', 0)
            row['有效率(%)'] = f"{v.get('validity_rate', 0):.1f}"
        
        # 优化统计
        if 'optimization_stats' in results:
            o = results['optimization_stats']
            row['优化成功'] = o.get('optimized', 0)
            row['优化率(%)'] = f"{o.get('success_rate', 0):.1f}"
        
        # 相似度统计
        if 'similarity_stats' in results:
            s = results['similarity_stats']
            if s.get('esp_mean') is not None:
                row['ESP相似度'] = f"{s['esp_mean']:.4f}±{s['esp_std']:.4f}"
                row['ESP最大'] = f"{s['esp_max']:.4f}"
            if s.get('pharm_mean') is not None:
                row['药效团相似度'] = f"{s['pharm_mean']:.4f}±{s['pharm_std']:.4f}"
                row['药效团最大'] = f"{s['pharm_max']:.4f}"
        
        # 筛选统计
        if 'filter_stats' in results:
            f = results['filter_stats']
            row['SA均值'] = f"{f.get('sa_mean', 0):.2f}" if f.get('sa_mean') else 'N/A'
            row['SA≤4.0(%)'] = f"{f.get('sa_pass_rate', 0):.1f}"
            row['中性电荷(%)'] = f"{f.get('neutral_rate', 0):.1f}"
            row['最终通过'] = f.get('filtered', 0)
            row['最终通过率(%)'] = f"{f.get('filter_rate', 0):.1f}"
        
        summary_data.append(row)
    
    return pd.DataFrame(summary_data)

# 创建汇总表
if all_results:
    summary_df = create_summary_table(all_results)
    print("\n" + "="*80)
    print("📊 Fragment Merging 评估结果汇总")
    print("="*80)
    display(summary_df)
else:
    print("⚠️ 无评估结果")

In [ ]:
# ==================== 详细属性表格 ====================

def create_detailed_properties_df(all_results):
    """创建所有分子的详细属性DataFrame"""
    all_props = []
    
    for model_name, results in all_results.items():
        if 'all_properties' in results:
            for prop in results['all_properties']:
                prop_copy = prop.copy()
                prop_copy['model'] = model_name
                all_props.append(prop_copy)
    
    if all_props:
        return pd.DataFrame(all_props)
    return None

# 创建详细属性表
if all_results:
    props_df = create_detailed_properties_df(all_results)
    if props_df is not None and len(props_df) > 0:
        print("\n" + "="*80)
        print("📋 所有分子详细属性 (前20行)")
        print("="*80)
        
        # 选择关键列显示
        display_cols = ['model', 'sample_idx', 'n_atoms', 'SA_score', 'QED', 'logP', 
                       'formal_charge', 'esp_similarity', 'pharm_similarity', 'smiles']
        available_cols = [c for c in display_cols if c in props_df.columns]
        display(props_df[available_cols].head(20))
        
        print(f"\n总计 {len(props_df)} 个分子")
    else:
        print("⚠️ 无详细属性数据")

In [ ]:
# ==================== 筛选后的高质量分子 ====================

def show_filtered_molecules(all_results):
    """显示通过筛选的高质量分子"""
    filtered_data = []
    
    for model_name, results in all_results.items():
        if 'filtered_samples' in results:
            for sample in results['filtered_samples']:
                if '_properties' in sample:
                    props = sample['_properties'].copy()
                    props['model'] = model_name
                    filtered_data.append(props)
    
    if filtered_data:
        return pd.DataFrame(filtered_data)
    return None

# 显示筛选后的分子
if all_results:
    filtered_df = show_filtered_molecules(all_results)
    if filtered_df is not None and len(filtered_df) > 0:
        print("\n" + "="*80)
        print(f"✨ 通过筛选的高质量分子 (SA≤{SA_THRESHOLD} 且 中性电荷)")
        print("="*80)
        
        display_cols = ['model', 'sample_idx', 'n_atoms', 'SA_score', 'QED', 'logP',
                       'esp_similarity', 'pharm_similarity', 'smiles']
        available_cols = [c for c in display_cols if c in filtered_df.columns]
        
        # 按ESP相似度排序
        if 'esp_similarity' in filtered_df.columns:
            filtered_df_sorted = filtered_df.sort_values('esp_similarity', ascending=False)
        else:
            filtered_df_sorted = filtered_df
        
        display(filtered_df_sorted[available_cols])
        
        print(f"\n✅ 共 {len(filtered_df)} 个分子通过筛选")
        
        # 按模型统计
        print("\n📊 各模型通过筛选数量:")
        for model in filtered_df['model'].unique():
            count = len(filtered_df[filtered_df['model'] == model])
            print(f"   - {model}: {count} 个")
    else:
        print("⚠️ 无通过筛选的分子")

In [ ]:
# ==================== 保存评估结果 ====================

def save_results(all_results, output_dir='.'):
    """保存评估结果到文件"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # 1. 保存汇总统计到JSON
    summary_data = {}
    for model_name, results in all_results.items():
        model_summary = {
            'model_name': model_name,
            'total_samples': results.get('total_samples', 0),
            'validity_stats': results.get('validity_stats', {}),
            'optimization_stats': results.get('optimization_stats', {}),
            'similarity_stats': results.get('similarity_stats', {}),
            'filter_stats': results.get('filter_stats', {}),
        }
        summary_data[model_name] = model_summary
    
    summary_file = os.path.join(output_dir, f'fragment_merge_eval_summary_{timestamp}.json')
    with open(summary_file, 'w', encoding='utf-8') as f:
        json.dump(summary_data, f, ensure_ascii=False, indent=2)
    print(f"✅ 汇总统计已保存: {summary_file}")
    
    # 2. 保存详细属性到CSV
    props_df = create_detailed_properties_df(all_results)
    if props_df is not None and len(props_df) > 0:
        props_file = os.path.join(output_dir, f'fragment_merge_all_properties_{timestamp}.csv')
        props_df.to_csv(props_file, index=False, encoding='utf-8')
        print(f"✅ 详细属性已保存: {props_file}")
    
    # 3. 保存筛选后分子到CSV
    filtered_df = show_filtered_molecules(all_results)
    if filtered_df is not None and len(filtered_df) > 0:
        filtered_file = os.path.join(output_dir, f'fragment_merge_filtered_{timestamp}.csv')
        filtered_df.to_csv(filtered_file, index=False, encoding='utf-8')
        print(f"✅ 筛选后分子已保存: {filtered_file}")
    
    return summary_file

# 保存结果
if all_results:
    output_dir = '/home1/zhh/workspace/SPD/evaluation/experiment_SamEval'
    save_results(all_results, output_dir)
    print("\n🎉 所有评估结果已保存完成!")

In [ ]:
# ==================== 可视化对比 ====================

import matplotlib.pyplot as plt

def plot_comparison(all_results):
    """绘制各模型评估结果对比图"""
    if not all_results:
        print("⚠️ 无数据可视化")
        return
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    models = list(all_results.keys())
    colors = plt.cm.Set2(np.linspace(0, 1, len(models)))
    
    # 1. 有效性和优化成功率对比
    ax1 = axes[0, 0]
    validity_rates = []
    opt_rates = []
    for model in models:
        v_stats = all_results[model].get('validity_stats', {})
        o_stats = all_results[model].get('optimization_stats', {})
        validity_rates.append(v_stats.get('validity_rate', 0))
        opt_rates.append(o_stats.get('success_rate', 0))
    
    x = np.arange(len(models))
    width = 0.35
    ax1.bar(x - width/2, validity_rates, width, label='有效率', color='steelblue')
    ax1.bar(x + width/2, opt_rates, width, label='优化成功率', color='coral')
    ax1.set_ylabel('百分比 (%)')
    ax1.set_title('有效性和优化成功率对比')
    ax1.set_xticks(x)
    ax1.set_xticklabels(models, rotation=15)
    ax1.legend()
    ax1.set_ylim(0, 105)
    
    # 2. ESP相似度对比
    ax2 = axes[0, 1]
    esp_means = []
    esp_stds = []
    for model in models:
        s_stats = all_results[model].get('similarity_stats', {})
        esp_means.append(s_stats.get('esp_mean', 0) or 0)
        esp_stds.append(s_stats.get('esp_std', 0) or 0)
    
    ax2.bar(models, esp_means, yerr=esp_stds, capsize=5, color=colors)
    ax2.set_ylabel('ESP相似度')
    ax2.set_title('ESP相似度对比')
    ax2.set_xticklabels(models, rotation=15)
    
    # 3. 药效团相似度对比
    ax3 = axes[1, 0]
    pharm_means = []
    pharm_stds = []
    for model in models:
        s_stats = all_results[model].get('similarity_stats', {})
        pharm_means.append(s_stats.get('pharm_mean', 0) or 0)
        pharm_stds.append(s_stats.get('pharm_std', 0) or 0)
    
    ax3.bar(models, pharm_means, yerr=pharm_stds, capsize=5, color=colors)
    ax3.set_ylabel('药效团相似度')
    ax3.set_title('药效团相似度对比')
    ax3.set_xticklabels(models, rotation=15)
    
    # 4. SA分数和筛选通过率
    ax4 = axes[1, 1]
    sa_pass_rates = []
    filter_rates = []
    for model in models:
        f_stats = all_results[model].get('filter_stats', {})
        sa_pass_rates.append(f_stats.get('sa_pass_rate', 0))
        filter_rates.append(f_stats.get('filter_rate', 0))
    
    ax4.bar(x - width/2, sa_pass_rates, width, label=f'SA≤{SA_THRESHOLD}', color='mediumseagreen')
    ax4.bar(x + width/2, filter_rates, width, label='SA≤4.0且中性', color='mediumpurple')
    ax4.set_ylabel('百分比 (%)')
    ax4.set_title('SA分数和最终筛选通过率')
    ax4.set_xticks(x)
    ax4.set_xticklabels(models, rotation=15)
    ax4.legend()
    ax4.set_ylim(0, 105)
    
    plt.tight_layout()
    plt.savefig('fragment_merge_evaluation_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✅ 对比图已保存: fragment_merge_evaluation_comparison.png")

# 绘制对比图
if all_results:
    plot_comparison(all_results)